In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
listing_path = Path(
    "../data/processed/shoes_with_adjusted_prices.parquet"
)

tier_path = Path(
    "../data/processed/brand_tiers.parquet"
)

shoes = pd.read_parquet(listing_path)
brand_tiers = pd.read_parquet(tier_path)

print("Listing-level shape:", shoes.shape)
print("Brand-tier shape:", brand_tiers.shape)

Listing-level shape: (53176, 12)
Brand-tier shape: (116, 5)


In [3]:
display(shoes.head())
display(brand_tiers.head())

,train_id,name,item_condition_id,category_name,brand_name,price,shipping,item_description,shoe_type,condition_group,peer_median_price,adjusted_price_ratio
0,14,HOLD for Dogs2016 Minnetonka boots,3,Women/Shoes/Boots,UGG Australia,43.0,0,Authentic. Suede fringe boots. Great condition...,Boots,Condition 3,36.0,1.194444
1,70,Adidas Ultraboost Shoes,3,Women/Shoes/Athletic,Adidas,61.0,0,Overall good condition. A few signs of wear,Athletic,Condition 3,28.0,2.178571
2,107,Boots NWT 6.5,1,Women/Shoes/Boots,Merona,13.0,1,Merona short boot new with tag size 6.5 come j...,Boots,Condition 1,61.0,0.213115
3,108,New Duck Boots sz.7.5,1,Women/Shoes/Boots,Boulevard Boutique,38.0,0,New Duck Boots Sz.7.5 Stock up on These Trendy...,Boots,Condition 1,61.0,0.622951
4,115,Steve Madden wedges,2,Women/Shoes/Pumps,Steve Madden,25.0,1,Never worn!!! Brown leather strap wedges!,Pumps,Condition 2,21.0,1.190476


,brand_name,listing_count,median_raw_price,median_adjusted_ratio,price_tier
0,ALDO,363,21.0,0.862069,Mid-Market
1,ASICS,335,26.0,0.850000,Mid-Market
2,Adidas,1965,51.0,1.464286,Premium
3,Aerosoles,84,16.5,0.726496,Mid-Market
4,American Eagle,500,14.0,0.571429,Value


In [4]:
shoes_with_tiers = shoes.merge(
    brand_tiers[
        [
            "brand_name",
            "price_tier"
        ]
    ],
    on="brand_name",
    how="inner"
)

print("Rows before merge:", len(shoes))
print("Rows after merge:", len(shoes_with_tiers))

shoes_with_tiers[
    [
        "brand_name",
        "shoe_type",
        "condition_group",
        "price",
        "adjusted_price_ratio",
        "price_tier"
    ]
].head(10)

Rows before merge: 53176
Rows after merge: 48685


,brand_name,shoe_type,condition_group,price,adjusted_price_ratio,price_tier
0,UGG Australia,Boots,Condition 3,43.0,1.194444,Premium
1,Adidas,Athletic,Condition 3,61.0,2.178571,Premium
2,Merona,Boots,Condition 1,13.0,0.213115,Value
3,Steve Madden,Pumps,Condition 2,25.0,1.190476,Mid-Market
4,GUESS,Sandals,Condition 1,31.0,1.068966,Mid-Market
5,JustFab,Boots,Condition 1,31.0,0.508197,Value
6,Crocs,Loafers & Slip-Ons,Condition 3,16.0,0.800000,Mid-Market
7,Coach,Fashion Sneakers,Condition 1,56.0,1.365854,Mid-Market
8,Tory Burch,Sandals,Condition 3,76.0,3.619048,Premium
9,Born,Sandals,Condition 3,16.0,0.761905,Mid-Market


In [5]:
shoes_with_tiers["tier_peer_median_ratio"] = (
    shoes_with_tiers
    .groupby(
        [
            "shoe_type",
            "condition_group",
            "price_tier"
        ]
    )["adjusted_price_ratio"]
    .transform("median")
)

shoes_with_tiers["resale_strength_ratio"] = (
    shoes_with_tiers["adjusted_price_ratio"]
    / shoes_with_tiers["tier_peer_median_ratio"]
)

shoes_with_tiers[
    [
        "brand_name",
        "shoe_type",
        "condition_group",
        "price_tier",
        "adjusted_price_ratio",
        "tier_peer_median_ratio",
        "resale_strength_ratio"
    ]
].head(10)

,brand_name,shoe_type,condition_group,price_tier,adjusted_price_ratio,tier_peer_median_ratio,resale_strength_ratio
0,UGG Australia,Boots,Condition 3,Premium,1.194444,1.527778,0.781818
1,Adidas,Athletic,Condition 3,Premium,2.178571,1.107143,1.967742
2,Merona,Boots,Condition 1,Value,0.213115,0.393443,0.541667
3,Steve Madden,Pumps,Condition 2,Mid-Market,1.190476,1.000000,1.190476
4,GUESS,Sandals,Condition 1,Mid-Market,1.068966,1.000000,1.068966
5,JustFab,Boots,Condition 1,Value,0.508197,0.393443,1.291667
6,Crocs,Loafers & Slip-Ons,Condition 3,Mid-Market,0.800000,1.000000,0.800000
7,Coach,Fashion Sneakers,Condition 1,Mid-Market,1.365854,0.926829,1.473684
8,Tory Burch,Sandals,Condition 3,Premium,3.619048,2.190476,1.652174
9,Born,Sandals,Condition 3,Mid-Market,0.761905,0.857143,0.888889


In [6]:
brand_resale_strength = (
    shoes_with_tiers
    .groupby(["brand_name", "price_tier"])
    .agg(
        listing_count=("price", "size"),
        resale_strength=("resale_strength_ratio", "median")
    )
    .reset_index()
)

brand_resale_strength.sort_values(
    "resale_strength",
    ascending=False
).head(20)

,brand_name,price_tier,listing_count,resale_strength
23,Christian Louboutin,Premium,301,3.304348
109,Valentino,Premium,55,3.000000
99,Tieks,Premium,59,2.964286
95,Stuart Weitzman,Premium,56,2.548341
20,Chanel,Premium,102,2.255137
44,Gucci,Premium,172,1.793637
16,Burberry,Premium,87,1.645161
100,Timberland,Mid-Market,445,1.483871
41,Frye,Premium,236,1.417143
53,Jordans,Premium,181,1.322581


In [7]:
brand_condition_strength = (
    shoes_with_tiers
    .groupby(
        [
            "brand_name",
            "price_tier",
            "condition_group"
        ]
    )
    .agg(
        listing_count=("price", "size"),
        resale_strength=("resale_strength_ratio", "median")
    )
    .reset_index()
)

brand_condition_strength.sort_values(
    ["brand_name", "condition_group"]
).head(30)

,brand_name,price_tier,condition_group,listing_count,resale_strength
0,ALDO,Mid-Market,Condition 1,41,0.909091
1,ALDO,Mid-Market,Condition 2,108,0.952381
2,ALDO,Mid-Market,Condition 3,193,0.888889
3,ALDO,Mid-Market,Condition 4–5,21,0.714286
4,ASICS,Mid-Market,Condition 1,43,0.819672
5,ASICS,Mid-Market,Condition 2,85,0.775000
6,ASICS,Mid-Market,Condition 3,187,0.857143
7,ASICS,Mid-Market,Condition 4–5,20,0.888889
8,Adidas,Premium,Condition 1,719,0.929577
9,Adidas,Premium,Condition 2,522,1.000000


In [9]:
brand_condition_strength[
    brand_condition_strength["brand_name"] == "Christian Louboutin"
]

,brand_name,price_tier,condition_group,listing_count,resale_strength
91,Christian Louboutin,Premium,Condition 1,52,1.682540
92,Christian Louboutin,Premium,Condition 2,67,2.983333
93,Christian Louboutin,Premium,Condition 3,157,3.941176
94,Christian Louboutin,Premium,Condition 4–5,25,3.448276


In [10]:
brand_price_consistency = (
    shoes_with_tiers
    .groupby(["brand_name", "price_tier"])
    .agg(
        listing_count=("price", "size"),
        median_price=("price", "median"),
        q1_price=("price", lambda x: x.quantile(0.25)),
        q3_price=("price", lambda x: x.quantile(0.75))
    )
    .reset_index()
)

brand_price_consistency["relative_price_spread"] = (
    (
        brand_price_consistency["q3_price"]
        - brand_price_consistency["q1_price"]
    )
    / brand_price_consistency["median_price"]
)

brand_price_consistency.sort_values(
    "relative_price_spread"
).head(20)

,brand_name,price_tier,listing_count,median_price,q1_price,q3_price,relative_price_spread
99,Tieks,Premium,59,156.0,148.00,171.00,0.147436
115,rue,Value,73,15.0,12.00,17.00,0.333333
86,Rue21,Value,89,14.0,11.00,16.00,0.357143
73,Mudd,Value,53,15.0,12.00,18.00,0.400000
55,JustFab,Value,83,20.0,16.00,24.00,0.400000
19,Chaco,Premium,281,56.0,43.00,66.00,0.410714
47,Hunter,Premium,586,77.0,59.25,94.00,0.451299
69,Minnetonka,Mid-Market,51,22.0,16.00,26.00,0.454545
70,Mizuno,Mid-Market,54,31.0,24.00,38.25,0.459677
5,Anne Klein,Mid-Market,58,17.0,14.00,22.00,0.470588


In [11]:
condition_medians = (
    shoes_with_tiers
    .groupby(
        [
            "brand_name",
            "price_tier",
            "condition_group"
        ]
    )
    .agg(
        listing_count=("price", "size"),
        median_price=("price", "median")
    )
    .reset_index()
)

condition_medians.head()

,brand_name,price_tier,condition_group,listing_count,median_price
0,ALDO,Mid-Market,Condition 1,41,28.0
1,ALDO,Mid-Market,Condition 2,108,24.0
2,ALDO,Mid-Market,Condition 3,193,20.0
3,ALDO,Mid-Market,Condition 4–5,21,12.0
4,ASICS,Mid-Market,Condition 1,43,50.0


In [12]:
condition_pivot = (
    condition_medians
    .pivot(
        index=["brand_name", "price_tier"],
        columns="condition_group",
        values="median_price"
    )
    .reset_index()
)

condition_pivot.head()

condition_group,brand_name,price_tier,Condition 1,Condition 2,Condition 3,Condition 4–5
0,ALDO,Mid-Market,28.0,24.0,20.0,12.0
1,ASICS,Mid-Market,50.0,31.0,24.0,16.0
2,Adidas,Premium,76.0,51.0,36.0,17.5
3,Aerosoles,Mid-Market,19.5,18.0,16.0,18.0
4,American Eagle,Value,18.0,15.0,12.0,12.0


In [13]:
condition_pivot["condition_resilience"] = (
    condition_pivot["Condition 3"]
    / condition_pivot["Condition 1"]
)

condition_pivot.sort_values(
    "condition_resilience",
    ascending=False
).head(20)

condition_group,brand_name,price_tier,Condition 1,Condition 2,Condition 3,Condition 4–5,condition_resilience
23,Christian Louboutin,Premium,106.0,161.0,204.0,109.0,1.924528
69,Minnetonka,Mid-Market,14.5,21.5,23.5,16.0,1.620690
20,Chanel,Premium,110.0,98.0,139.0,62.0,1.263636
50,Jack Rogers,Premium,35.0,35.0,36.0,22.0,1.028571
59,L.L. Bean,Premium,68.5,86.0,70.0,36.0,1.021898
31,Dollhouse,Value,15.0,16.0,15.0,8.0,1.000000
43,Gap,Value,16.0,11.0,16.0,8.0,1.000000
45,H&M,Value,14.0,16.0,14.0,16.0,1.000000
10,Betsey Johnson,Mid-Market,22.0,25.5,21.5,11.0,0.977273
22,Chinese Laundry,Mid-Market,19.5,22.0,19.0,12.0,0.974359


In [14]:
category_condition_medians = (
    shoes_with_tiers
    .groupby(
        [
            "brand_name",
            "price_tier",
            "shoe_type",
            "condition_group"
        ]
    )
    .agg(
        listing_count=("price", "size"),
        median_price=("price", "median")
    )
    .reset_index()
)

category_condition_medians.head(10)

,brand_name,price_tier,shoe_type,condition_group,listing_count,median_price
0,ALDO,Mid-Market,Boots,Condition 1,13,46.0
1,ALDO,Mid-Market,Boots,Condition 2,31,31.0
2,ALDO,Mid-Market,Boots,Condition 3,52,23.0
3,ALDO,Mid-Market,Boots,Condition 4–5,9,12.0
4,ALDO,Mid-Market,Fashion Sneakers,Condition 1,2,53.5
5,ALDO,Mid-Market,Fashion Sneakers,Condition 2,9,31.0
6,ALDO,Mid-Market,Fashion Sneakers,Condition 3,14,20.5
7,ALDO,Mid-Market,Flats,Condition 1,7,23.0
8,ALDO,Mid-Market,Flats,Condition 2,8,20.5
9,ALDO,Mid-Market,Flats,Condition 3,21,15.0


In [15]:
category_condition_pivot = (
    category_condition_medians
    .pivot(
        index=[
            "brand_name",
            "price_tier",
            "shoe_type"
        ],
        columns="condition_group",
        values="median_price"
    )
    .reset_index()
)

category_condition_pivot.head(10)

condition_group,brand_name,price_tier,shoe_type,Condition 1,Condition 2,Condition 3,Condition 4–5
0,ALDO,Mid-Market,Boots,46.0,31.0,23.0,12.0
1,ALDO,Mid-Market,Fashion Sneakers,53.5,31.0,20.5,NaN
2,ALDO,Mid-Market,Flats,23.0,20.5,15.0,8.0
3,ALDO,Mid-Market,Loafers & Slip-Ons,NaN,20.0,20.5,NaN
4,ALDO,Mid-Market,Pumps,31.0,20.0,20.0,14.0
5,ALDO,Mid-Market,Sandals,22.5,21.5,16.0,7.0
6,ASICS,Mid-Market,Athletic,48.0,31.0,24.0,16.0
7,ASICS,Mid-Market,Fashion Sneakers,75.0,29.5,23.0,NaN
8,ASICS,Mid-Market,Flats,NaN,NaN,10.0,NaN
9,Adidas,Premium,Athletic,106.0,56.0,33.0,19.0


In [16]:
category_condition_pivot["category_resilience"] = (
    category_condition_pivot["Condition 3"]
    / category_condition_pivot["Condition 1"]
)

category_condition_pivot[
    [
        "brand_name",
        "price_tier",
        "shoe_type",
        "Condition 1",
        "Condition 3",
        "category_resilience"
    ]
].head(20)

condition_group,brand_name,price_tier,shoe_type,Condition 1,Condition 3,category_resilience
0,ALDO,Mid-Market,Boots,46.0,23.0,0.500000
1,ALDO,Mid-Market,Fashion Sneakers,53.5,20.5,0.383178
2,ALDO,Mid-Market,Flats,23.0,15.0,0.652174
3,ALDO,Mid-Market,Loafers & Slip-Ons,NaN,20.5,NaN
4,ALDO,Mid-Market,Pumps,31.0,20.0,0.645161
5,ALDO,Mid-Market,Sandals,22.5,16.0,0.711111
6,ASICS,Mid-Market,Athletic,48.0,24.0,0.500000
7,ASICS,Mid-Market,Fashion Sneakers,75.0,23.0,0.306667
8,ASICS,Mid-Market,Flats,NaN,10.0,NaN
9,Adidas,Premium,Athletic,106.0,33.0,0.311321


In [17]:
category_condition_counts = (
    category_condition_medians
    .pivot(
        index=[
            "brand_name",
            "price_tier",
            "shoe_type"
        ],
        columns="condition_group",
        values="listing_count"
    )
    .reset_index()
)

category_condition_counts = category_condition_counts.rename(
    columns={
        "Condition 1": "condition_1_count",
        "Condition 3": "condition_3_count"
    }
)

category_condition_counts.head()

condition_group,brand_name,price_tier,shoe_type,condition_1_count,Condition 2,condition_3_count,Condition 4–5
0,ALDO,Mid-Market,Boots,13.0,31.0,52.0,9.0
1,ALDO,Mid-Market,Fashion Sneakers,2.0,9.0,14.0,NaN
2,ALDO,Mid-Market,Flats,7.0,8.0,21.0,1.0
3,ALDO,Mid-Market,Loafers & Slip-Ons,NaN,3.0,2.0,NaN
4,ALDO,Mid-Market,Pumps,11.0,41.0,78.0,9.0


In [18]:
category_resilience = category_condition_pivot.merge(
    category_condition_counts[
        [
            "brand_name",
            "price_tier",
            "shoe_type",
            "condition_1_count",
            "condition_3_count"
        ]
    ],
    on=[
        "brand_name",
        "price_tier",
        "shoe_type"
    ],
    how="left"
)

In [19]:
reliable_category_resilience = category_resilience[
    (category_resilience["condition_1_count"] >= 10)
    & (category_resilience["condition_3_count"] >= 10)
    & category_resilience["category_resilience"].notna()
].copy()

print(
    "Reliable brand-category comparisons:",
    len(reliable_category_resilience)
)

Reliable brand-category comparisons: 132


In [20]:
brand_condition_resilience = (
    reliable_category_resilience
    .groupby(["brand_name", "price_tier"])
    .agg(
        condition_resilience=(
            "category_resilience",
            "median"
        ),
        qualifying_shoe_types=(
            "shoe_type",
            "nunique"
        )
    )
    .reset_index()
)

brand_condition_resilience.sort_values(
    "condition_resilience",
    ascending=False
).head(20)

,brand_name,price_tier,condition_resilience,qualifying_shoe_types
11,Christian Louboutin,Premium,2.122642,1
26,Jack Rogers,Premium,1.043478,1
30,L.L. Bean,Premium,1.000000,1
20,Free People,Premium,0.966138,2
8,Chaco,Premium,0.925620,1
33,Merona,Value,0.878401,2
40,Old Navy,Value,0.857143,2
31,Lilly Pulitzer,Premium,0.857143,1
23,Gucci,Premium,0.850000,1
63,Wet Seal,Value,0.804348,2


In [21]:
brand_metrics = (
    brand_resale_strength
    .merge(
        brand_price_consistency[
            [
                "brand_name",
                "price_tier",
                "median_price",
                "relative_price_spread"
            ]
        ],
        on=["brand_name", "price_tier"],
        how="left"
    )
    .merge(
        brand_condition_resilience[
            [
                "brand_name",
                "price_tier",
                "condition_resilience",
                "qualifying_shoe_types"
            ]
        ],
        on=["brand_name", "price_tier"],
        how="left"
    )
)

brand_metrics.head(10)

,brand_name,price_tier,listing_count,resale_strength,median_price,relative_price_spread,condition_resilience,qualifying_shoe_types
0,ALDO,Mid-Market,363,0.920000,21.0,0.666667,0.572581,2.0
1,ASICS,Mid-Market,335,0.850000,26.0,0.692308,0.500000,1.0
2,Adidas,Premium,1965,0.975610,51.0,1.098039,0.500000,3.0
3,Aerosoles,Mid-Market,84,0.761905,16.5,0.606061,NaN,NaN
4,American Eagle,Value,500,1.000000,14.0,0.642857,0.772727,3.0
5,Anne Klein,Mid-Market,58,0.821053,17.0,0.470588,NaN,NaN
6,Ariat,Premium,207,0.842857,51.0,0.529412,0.676056,1.0
7,BCBGeneration,Mid-Market,167,0.888889,20.0,0.700000,0.484848,1.0
8,Bandolino,Mid-Market,54,0.783626,17.0,0.558824,NaN,NaN
9,Bebe,Mid-Market,79,1.137931,26.0,0.673077,NaN,NaN


In [22]:
print("Total brands:", len(brand_metrics))

print(
    "Brands with condition resilience:",
    brand_metrics["condition_resilience"].notna().sum()
)

print(
    "Brands without condition resilience:",
    brand_metrics["condition_resilience"].isna().sum()
)

Total brands: 116
Brands with condition resilience: 67
Brands without condition resilience: 49


In [23]:
brand_metrics["resale_strength_score"] = (
    brand_metrics
    .groupby("price_tier")["resale_strength"]
    .rank(pct=True)
    * 100
)

brand_metrics["consistency_score"] = (
    brand_metrics
    .groupby("price_tier")["relative_price_spread"]
    .rank(pct=True, ascending=False)
    * 100
)

brand_metrics["resilience_score"] = (
    brand_metrics
    .groupby("price_tier")["condition_resilience"]
    .rank(pct=True)
    * 100
)

In [24]:
brand_metrics[
    [
        "brand_name",
        "price_tier",
        "resale_strength_score",
        "consistency_score",
        "resilience_score"
    ]
].head(15)

,brand_name,price_tier,resale_strength_score,consistency_score,resilience_score
0,ALDO,Mid-Market,50.877193,36.842105,39.393939
1,ASICS,Mid-Market,35.087719,28.947368,18.181818
2,Adidas,Premium,41.379310,27.586207,8.695652
3,Aerosoles,Mid-Market,7.017544,57.894737,NaN
4,American Eagle,Value,45.000000,15.000000,63.636364
5,Anne Klein,Mid-Market,24.561404,96.491228,NaN
6,Ariat,Premium,24.137931,86.206897,43.478261
7,BCBGeneration,Mid-Market,43.859649,25.438596,12.121212
8,Bandolino,Mid-Market,12.280702,70.175439,NaN
9,Bebe,Mid-Market,82.456140,33.333333,NaN


In [25]:
complete_brand_metrics = brand_metrics.dropna(
    subset=[
        "resale_strength_score",
        "consistency_score",
        "resilience_score"
    ]
).copy()

complete_brand_metrics["overall_score"] = (
    complete_brand_metrics[
        [
            "resale_strength_score",
            "consistency_score",
            "resilience_score"
        ]
    ].mean(axis=1)
)

print(
    "Brands with complete overall scores:",
    len(complete_brand_metrics)
)

complete_brand_metrics.sort_values(
    ["price_tier", "overall_score"],
    ascending=[True, False]
)[
    [
        "brand_name",
        "price_tier",
        "listing_count",
        "resale_strength_score",
        "consistency_score",
        "resilience_score",
        "overall_score",
        "qualifying_shoe_types"
    ]
].head(20)

Brands with complete overall scores: 67


,brand_name,price_tier,listing_count,resale_strength_score,consistency_score,resilience_score,overall_score,qualifying_shoe_types
52,Jessica Simpson,Mid-Market,335,57.894737,80.701754,100.000000,79.532164,1.0
79,PINK,Mid-Market,1160,75.438596,91.228070,71.212121,79.292929,1.0
26,Converse,Mid-Market,3204,59.649123,86.842105,90.909091,79.133440,2.0
27,Converse Shoes,Mid-Market,141,40.350877,86.842105,96.969697,74.720893,1.0
90,Sanuk,Mid-Market,123,49.122807,86.842105,87.878788,74.614567,1.0
24,Coach,Mid-Market,1391,86.842105,54.385965,63.636364,68.288145,6.0
89,Sam Edelman,Mid-Market,259,89.473684,36.842105,71.212121,65.842637,3.0
98,The North Face,Mid-Market,67,80.701754,44.736842,57.575758,61.004785,1.0
93,Sperrys,Mid-Market,339,70.175439,67.543860,43.939394,60.552897,1.0
28,Crocs,Mid-Market,389,1.754386,86.842105,84.848485,57.814992,2.0


In [26]:
complete_brand_metrics["tier_rank"] = (
    complete_brand_metrics
    .groupby("price_tier")["overall_score"]
    .rank(method="min", ascending=False)
    .astype(int)
)

In [27]:
complete_brand_metrics.sort_values(
    ["price_tier", "tier_rank"]
)[
    [
        "tier_rank",
        "brand_name",
        "price_tier",
        "listing_count",
        "resale_strength_score",
        "consistency_score",
        "resilience_score",
        "overall_score",
        "qualifying_shoe_types"
    ]
].head(30)

,tier_rank,brand_name,price_tier,listing_count,resale_strength_score,consistency_score,resilience_score,overall_score,qualifying_shoe_types
52,1,Jessica Simpson,Mid-Market,335,57.894737,80.701754,100.000000,79.532164,1.0
79,2,PINK,Mid-Market,1160,75.438596,91.228070,71.212121,79.292929,1.0
26,3,Converse,Mid-Market,3204,59.649123,86.842105,90.909091,79.133440,2.0
27,4,Converse Shoes,Mid-Market,141,40.350877,86.842105,96.969697,74.720893,1.0
90,5,Sanuk,Mid-Market,123,49.122807,86.842105,87.878788,74.614567,1.0
24,6,Coach,Mid-Market,1391,86.842105,54.385965,63.636364,68.288145,6.0
89,7,Sam Edelman,Mid-Market,259,89.473684,36.842105,71.212121,65.842637,3.0
98,8,The North Face,Mid-Market,67,80.701754,44.736842,57.575758,61.004785,1.0
93,9,Sperrys,Mid-Market,339,70.175439,67.543860,43.939394,60.552897,1.0
28,10,Crocs,Mid-Market,389,1.754386,86.842105,84.848485,57.814992,2.0


In [28]:
def assign_evidence_level(qualifying_shoe_types):
    if qualifying_shoe_types >= 4:
        return "Strong"
    elif qualifying_shoe_types >= 2:
        return "Moderate"
    else:
        return "Limited"

complete_brand_metrics["evidence_level"] = (
    complete_brand_metrics["qualifying_shoe_types"]
    .apply(assign_evidence_level)
)

In [29]:
complete_brand_metrics.sort_values(
    ["price_tier", "tier_rank"]
)[
    [
        "tier_rank",
        "brand_name",
        "price_tier",
        "overall_score",
        "resale_strength_score",
        "consistency_score",
        "resilience_score",
        "qualifying_shoe_types",
        "evidence_level"
    ]
].head(30)

,tier_rank,brand_name,price_tier,overall_score,resale_strength_score,consistency_score,resilience_score,qualifying_shoe_types,evidence_level
52,1,Jessica Simpson,Mid-Market,79.532164,57.894737,80.701754,100.000000,1.0,Limited
79,2,PINK,Mid-Market,79.292929,75.438596,91.228070,71.212121,1.0,Limited
26,3,Converse,Mid-Market,79.133440,59.649123,86.842105,90.909091,2.0,Moderate
27,4,Converse Shoes,Mid-Market,74.720893,40.350877,86.842105,96.969697,1.0,Limited
90,5,Sanuk,Mid-Market,74.614567,49.122807,86.842105,87.878788,1.0,Limited
24,6,Coach,Mid-Market,68.288145,86.842105,54.385965,63.636364,6.0,Strong
89,7,Sam Edelman,Mid-Market,65.842637,89.473684,36.842105,71.212121,3.0,Moderate
98,8,The North Face,Mid-Market,61.004785,80.701754,44.736842,57.575758,1.0,Limited
93,9,Sperrys,Mid-Market,60.552897,70.175439,67.543860,43.939394,1.0,Limited
28,10,Crocs,Mid-Market,57.814992,1.754386,86.842105,84.848485,2.0,Moderate


In [30]:
metrics_output_path = Path(
    "../data/processed/final_brand_metrics.parquet"
)

complete_brand_metrics.to_parquet(
    metrics_output_path,
    index=False
)

print("Saved to:", metrics_output_path)
print("Saved brands:", len(complete_brand_metrics))

Saved to: ../data/processed/final_brand_metrics.parquet
Saved brands: 67


In [31]:
csv_output_path = Path(
    "../outputs/tables/final_brand_metrics.csv"
)

complete_brand_metrics.to_csv(
    csv_output_path,
    index=False
)

print("CSV saved to:", csv_output_path)

CSV saved to: ../outputs/tables/final_brand_metrics.csv
